# 03 · Reinforcement learning: Gymnasium + pure JAX

**What this notebook is.** A tour of the two reinforcement-learning surfaces the
library exposes. **The library implements no RL algorithm** — it gives you the
environment, the observation/action spaces and the reward; a trainer
(Stable-Baselines3, CleanRL, your own loop) supplies the learning.

**What you will do**

1. wrap a tiny 2-crease pattern in `OrigamiEnv` and inspect its spaces;
2. run Gymnasium's `check_env` conformance check;
3. roll out a random policy and read the reward/info contract;
4. drive the same machinery as **pure JAX** (`build_jax_model` → `reset` →
   jitted `step` → `lax.scan`), which is the path to `vmap` / GPU training;
5. optionally delegate the learning to Stable-Baselines3 PPO.

> ### Cost warning (read this once)
>
> Every `step()` is real MJX contact physics. This 6-crease demo is the
> cheapest pattern with more than one action, but a CPU runtime still spends
> seconds per step, and the jitted step takes tens of seconds to compile. `RUN_VECTOR` and `RUN_SB3` are
> therefore **off by default**; enable them on a GPU runtime with a tiny budget.
> The notebook never claims CPU speed-ups from batching.


## Install

`[jax,rl]` pulls in `mujoco-mjx`/`jax` and `gymnasium`. After the integration
branch is merged, replace `@feat/optimization-integrations-v1` with `@main`.


In [ ]:
%pip install -q "origami-engine-mjx[jax,rl] @ git+https://github.com/ihsan31415/origami-engine-mjx.git@feat/optimization-integrations-v1"


## 0. Configure

`MAX_STEPS` is the episode horizon (one `step` can run `frame_skip` physics
sub-steps). Keep it tiny on CPU.


In [ ]:
import numpy as np

from origami_engine_mjx import Crease
from origami_engine_mjx.integrations import SearchSpace, save_genome
from origami_engine_mjx.integrations.env import OrigamiEnv

FAST_DEMO = True
MAX_STEPS = 2 if FAST_DEMO else 20
N_ENVS = 2
VEC_STEPS = 2

RUN_CHECK_ENV = True     # Gymnasium conformance check (one short episode)
RUN_VECTOR = False       # vmapped vector env: useful on GPU, slow on CPU
RUN_SB3 = False          # Stable-Baselines3 PPO: installs torch, needs a GPU/real budget

print("max_steps:", MAX_STEPS)


## 1. A fixed-topology environment

RL needs **one crease topology**: the model is compiled once and only the state
changes. `OrigamiEnv` accepts a `SearchSpace` or an angle-only `DesignSpace`;
geometry / crease-count design spaces are rejected with a clear `ValueError`.

The 2×2 Miura pattern has 6 creases and **4 actuated hinges**, so the action is
a 4-vector and the observation is `dim + 3 = 7` floats: normalized target
angles, centroid x/y and episode progress.


In [ ]:
creases = [
    Crease((0.1, 0.1), (0.1, 0.5), 0.8),
    Crease((0.1, 0.5), (0.5, 0.5), -0.8),
    Crease((0.5, 0.5), (0.5, 0.1), 0.8),
    Crease((0.5, 0.1), (0.1, 0.1), -0.8),
    Crease((0.1, 0.1), (0.5, 0.5), -0.8),
    Crease((0.1, 0.5), (0.5, 0.1), 0.8),
]
space = SearchSpace.from_creases(creases)
env = OrigamiEnv(space=space, max_steps=MAX_STEPS, frame_skip=1, vibrate=False)

print("space dim        :", space.dim)
print("observation space:", env.observation_space)
print("action space     :", env.action_space)
print("metadata         :", {k: v for k, v in env.metadata.items() if k != "render_modes"})


## 2. Gymnasium conformance

`check_env` exercises `reset`/`step`, the spaces and the `terminated` /
`truncated` flags. It does run physics, so the check env uses the shortest
possible horizon.


In [ ]:
if RUN_CHECK_ENV:
    from gymnasium.utils.env_checker import check_env

    probe = OrigamiEnv(space=space, max_steps=2, frame_skip=1, vibrate=False)
    check_env(probe, skip_render_check=True)
    probe.close()
    print("check_env passed")
else:
    print("check_env skipped")


## 3. Roll out a random policy

* **action** — shape `(dim,)` in `[-1, 1]` by default (`normalized_action=True`);
  mapped linearly onto `[lower, upper]`. Use `action_mode="delta"` for relative
  moves.
* **reward** — the *change* in +x centroid displacement, so the episode return
  equals the total forward displacement in metres.
* **termination** — `terminated` only on non-finite physics (`status="unstable"`);
  `truncated` at `max_steps`.


In [ ]:
obs, info = env.reset(seed=0)
print("reset obs :", np.round(obs, 4).tolist())
print("reset info:", info)

rng = np.random.default_rng(0)
total = 0.0
for _ in range(env.max_steps):
    action = rng.uniform(-1.0, 1.0, size=env.action_space.shape).astype(np.float32)
    obs, reward, terminated, truncated, info = env.step(action)
    total += reward
    print(f"step {info['step_count']}: reward={reward:+.4f} m  "
          f"forward={info['forward_displacement']:+.4f} m  status={info['status']}")
    if terminated or truncated:
        break

print(f"episode return: {total:+.4f} m")
print("final targets:", np.round(info["targets"], 4).tolist())


## 4. Save the final folded design

`info["targets"]` are fold angles in genome order, so they can be written as an
ordinary crease-pattern JSON with `save_genome`.


In [ ]:
path = save_genome(space.repair(info["targets"]), space, "runs/rl_targets.json")
print("wrote", path)
env.close()


## 5. The same environment as pure JAX functions

`integrations.jax_env` exposes `build_jax_model`, `make_params`, `reset`,
`step` and `observe` as pure functions over pytrees. This is what `vmap` /
`scan` / GPU training consume; there is no Python loop in the hot path.

JIT the step **once**, outside the training loop — tracing this contact model is
expensive.


In [ ]:
import jax
import jax.numpy as jnp

from origami_engine_mjx.integrations import jax_env

model, classic, model_info = jax_env.build_jax_model(space.decode(np.zeros(space.dim)))
params = jax_env.make_params(
    space,
    classic,
    model_info,
    frame_skip=1,
    max_steps=MAX_STEPS,
    vibrate=False,
)
state = jax_env.reset(None, model, params)

step_fn = jax.jit(lambda s, a: jax_env.step(model, s, params, a))
state, obs, reward, done = step_fn(state, jnp.zeros(space.dim, dtype=jnp.float32))

print("obs       :", np.round(np.asarray(obs), 4).tolist())
print("reward    :", f"{float(reward):+.4f} m")
print("done      :", bool(done))
print("state keys:", state._fields)


## 6. A compiled rollout with `lax.scan`

`scan` replaces the Python loop, so the whole rollout becomes one compiled
program (the key primitive for on-device RL). Compilation happens on the first
call.


In [ ]:
actions = jnp.asarray(
    np.random.default_rng(0).uniform(-1.0, 1.0, size=(MAX_STEPS, space.dim)),
    dtype=jnp.float32,
)


def rollout(actions):
    state = jax_env.reset(None, model, params)

    def body(carry, action):
        carry, obs, reward, done = step_fn(carry, action)
        return carry, (obs, reward, done)

    state, (obs_seq, rewards, dones) = jax.lax.scan(body, state, actions)
    return state, obs_seq, rewards, dones


state, obs_seq, rewards, dones = jax.jit(rollout)(actions)
print("obs sequence shape:", obs_seq.shape)
print("rewards           :", np.round(np.asarray(rewards), 4).tolist())
print("cumulative return :", f"{float(jnp.sum(rewards)):+.4f} m")
print("dones             :", np.asarray(dones).tolist())


## 7. Vectorized environments (optional)

`OrigamiVectorEnv` steps `N` environments with one `jax.vmap`, sharing the
compiled model. The API mirrors `gymnasium.vector`: `reset() -> (obs, info)` and
`step(actions) -> (obs, rewards, terminated, truncated, info)` with a leading
`N` axis.

The vmapped step is expensive to compile and is aimed at GPU workloads; on CPU
it is dominated by `mjx.step`. This cell is off by default.


In [ ]:
if RUN_VECTOR:
    from origami_engine_mjx.integrations.env import OrigamiVectorEnv

    venv = OrigamiVectorEnv(space, n_envs=N_ENVS, max_steps=VEC_STEPS, frame_skip=1)
    obs, info = venv.reset(seed=0)
    print("reset   :", obs.shape, info)

    actions = np.random.default_rng(1).uniform(-1.0, 1.0, size=(N_ENVS, space.dim)).astype(np.float32)
    for _ in range(VEC_STEPS):
        obs, rewards, terminated, truncated, info = venv.step(actions)
        print("rewards :", np.round(rewards, 4).tolist(), "| truncated:", truncated.tolist())
else:
    print("RUN_VECTOR is False. Set it to True on a GPU runtime.")


## 8. Hand the environment to an RL library (optional)

This is the intended boundary: `OrigamiEnv` is a plain Gymnasium env, so any
trainer works. The cell below uses Stable-Baselines3 PPO — **an external
package**, not part of `origami_engine_mjx`.

Install it separately (it pulls in PyTorch, a large download):

```python
%pip install -q stable-baselines3
```

Then set `RUN_SB3 = True`. Budget honestly: `total_timesteps` full physics
steps, so start with a handful and a GPU runtime.


In [ ]:
if RUN_SB3:
    from stable_baselines3 import PPO
    from stable_baselines3.common.monitor import Monitor
    from stable_baselines3.common.vec_env import DummyVecEnv

    def make_env():
        return Monitor(OrigamiEnv(space=space, max_steps=MAX_STEPS, frame_skip=1))

    train_env = DummyVecEnv([make_env])
    agent = PPO(
        "MlpPolicy",
        train_env,
        n_steps=4,
        batch_size=4,
        n_epochs=1,
        learning_rate=3e-4,
        verbose=1,
        seed=0,
    )
    agent.learn(total_timesteps=8)

    obs = train_env.reset()
    action, _ = agent.predict(obs, deterministic=True)
    print("policy action:", np.round(action, 4).tolist())
    train_env.close()
else:
    print("RUN_SB3 is False. Install stable-baselines3, set RUN_SB3 = True, and use a GPU runtime.")


## 9. Pitfalls

| Topic | Rule |
|---|---|
| Topology | RL needs one fixed crease set; use a `SearchSpace` or angle-only `DesignSpace` |
| Action modes | `normalized_action=True` → `[-1, 1]`; `action_mode="delta"` moves relative to the current targets |
| Reward | Default is the step change in +x displacement; episode return = total displacement |
| Termination | `terminated` only on non-finite physics; `truncated` at `max_steps` |
| Seeding | Physics is deterministic; `reset(seed=...)` only matters with `options={"randomize_start": True}` |
| Cost | Each step is full MJX contact physics; compile `jax_env.step` once and reuse it |
| Differentiability | Contacts are non-smooth; do not rely on gradients through `step` |
| Batching | `OrigamiVectorEnv` is a GPU-oriented `vmap` path, not a CPU speed-up claim |
| Algorithms | The library ships **none**; SB3 / CleanRL / your loop owns the learning |

Backends: the RL envs wrap the **MJX-JAX** backend only; Warp is stateful and is
not wrapped.

Next: **04** (an AI / LLM agent in the loop) · **01** (black-box optimizers) ·
**02** (every `evosax` algorithm).

Docs: `docs/RL.md` · `docs/INTEGRATIONS.md` · `docs/COLAB_GPU.md`.
Reusable recipe: `prompts/origami-optimizer-integration.md`.
